# Use ONNX model converted from AutoAI with `ibm-watsonx-ai`

This notebook facilitates `ONNX`, `AutoAI`, and `watsonx.ai Runtime` service. It contains steps and code to work with [ibm-watsonx-ai](https://pypi.python.org/pypi/ibm-watsonx-ai) library available in PyPI repository in order to convert the model to ONNX format. It also introduces commands for persisting, deploying and scoring the model.

Some familiarity with Python is helpful. This notebook uses Python 3.11.

## Learning goals

The learning goals of this notebook are:

-  Train an AutoAI model
-  Convert the native scikit-learn model to ONNX format
-  Deploy the model for online scoring using client library
-  Score sample records using the client library

## Contents

This notebook contains the following parts:

1. [Set up the environment](#1.-Set-up-the-environment)
2. [Optimizer definition](#2.-Optimizer-definition)
3. [Experiment run](#3.-Experiment-run)
4. [Deploy and score](#4.-Deploy-and-score)
5. [Cleanup](#5.-Cleanup)
6. [Summary and next steps](#6.-Summary-and-next-steps)

<a id="1.-Set-up-the-environment"></a>
## 1. Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a [watsonx.ai Runtime](https://cloud.ibm.com/catalog/services/watsonxai-runtime) instance (information on service plans and further reading can be found [here](https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp)).

### 1.1. Installing and importing the `ibm-watsonx-ai` and dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install ibm-watsonx-ai | tail -n 1
%pip install wget | tail -n 1
%pip install "autoai_libs[onnx]<3.0.0" | tail -n 1
%pip install onnxruntime | tail -n 1

### 1.2. Connecting to watsonx.ai Runtime

Authenticate with the watsonx.ai Runtime service on IBM Cloud. You need to provide platform `api_key` and instance `location`.

You can use [IBM Cloud CLI](https://cloud.ibm.com/docs/cli/index.html) to retrieve platform API Key and instance location.

API Key can be generated in the following way:
```
ibmcloud login
ibmcloud iam api-key-create API_KEY_NAME
```

Get the value of `api_key` from the output.


Location of your watsonx.ai Runtime instance can be retrieved in the following way:
```
ibmcloud login --apikey API_KEY -a https://cloud.ibm.com
ibmcloud resource service-instance INSTANCE_NAME
```

Get the value of `location` from the output.

**Tip**: You can generate your `Cloud API key` by going to the [**Users** section of the Cloud console](https://cloud.ibm.com/iam#/users). From that page, click your name, scroll down to the **API Keys** section, and click **Create an IBM Cloud API key**. Give your key a name and click **Create**, then copy the created key and paste it below. You can also get a service-specific url by going to the [**Endpoint URLs** section of the watsonx.ai Runtime docs](https://cloud.ibm.com/apidocs/machine-learning).  You can check your instance location in your  <a href="https://cloud.ibm.com/catalog/services/watson-machine-learning" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance details.

You can also get the service specific apikey by going to the [**Service IDs** section of the Cloud Console](https://cloud.ibm.com/iam/serviceids).  From that page, click **Create**, then copy the created key, and paste it below.

**Action**: Enter your `api_key` and `location` in the following cells.

In [2]:
import getpass

api_key = getpass.getpass("Please enter your api key (hit enter): ")

In [3]:
location = "PASTE YOUR LOCATION HERE"

If you are running this notebook on Cloud, you can access the `location` via:

```
location = os.environ.get("RUNTIME_ENV_REGION")
```

In [4]:
from ibm_watsonx_ai import Credentials

credentials = Credentials(api_key=api_key, url=f"https://{location}.ml.cloud.ibm.com")

In [5]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials=credentials)

### 1.3. Working with spaces

First of all, you need to create a space that will be used for your work. If you do not have a space, you can use [Deployment Spaces Dashboard](https://dataplatform.cloud.ibm.com/ml-runtime/spaces?context=cpdaas) to create one.

- Click New Deployment Space
- Create an empty space
- Select Cloud Object Storage
- Select watsonx.ai Runtime instance and press Create
- Copy `space_id` and paste it below

**Tip**: You can also use the `ibm_watsonx_ai` SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: Assign space ID below

In [6]:
space_id = "PASTE YOUR SPACE ID HERE"

You can use the `list` method to print all existing spaces.

In [ ]:
client.spaces.list(limit=10)

To be able to interact with all resources available in watsonx.ai Runtime, you need to set **space** which you will be using.

In [7]:
client.set.default_space(space_id)

'SUCCESS'

### Connections to COS

In next cell we read the COS credentials from the space.

In [8]:
from ibm_watsonx_ai.utils import get_from_json

In [9]:
space_details = client.spaces.get_details(space_id=space_id)

cos_credentials = get_from_json(space_details, ["entity", "storage", "properties"])

<a id="2.-Optimizer-definition"></a>
## 2. Optimizer definition

### Training data connection

Define connection information to COS bucket and training data CSV file. This example uses the [German Credit Risk dataset](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/data/credit_risk/credit_risk_training_light.csv).

The code in next cell uploads training data to the bucket.

In [10]:
filename = "credit_risk_training_light.csv"
datasource_name = "bluemixcloudobjectstorage"
bucket_name = cos_credentials.get("bucket_name")

Download training data from git repository.

In [11]:
import os

import wget

url = "https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/data/credit_risk/credit_risk_training_light.csv"
if not os.path.isfile(filename):
    wget.download(url)

#### Create connection

In [12]:
conn_meta_props = {
    client.connections.ConfigurationMetaNames.NAME: f"Connection to Database - {datasource_name} ",
    client.connections.ConfigurationMetaNames.DATASOURCE_TYPE: client.connections.get_datasource_type_id_by_name(
        datasource_name
    ),
    client.connections.ConfigurationMetaNames.DESCRIPTION: "Connection to external Database",
    client.connections.ConfigurationMetaNames.PROPERTIES: {
        "bucket": bucket_name,
        "access_key": get_from_json(
            cos_credentials, ["credentials", "editor", "access_key_id"]
        ),
        "secret_key": get_from_json(
            cos_credentials, ["credentials", "editor", "secret_access_key"]
        ),
        "iam_url": "https://iam.cloud.ibm.com/identity/token",
        "url": cos_credentials.get("endpoint_url"),
    },
}

conn_details = client.connections.create(meta_props=conn_meta_props)

Creating connections...
SUCCESS


**Note**: The above connection can be initialized alternatively with `api_key` and `resource_instance_id`.  
The above cell can be replaced with:


```
conn_meta_props= {
    client.connections.ConfigurationMetaNames.NAME: f"Connection to Database - {db_name} ",
    client.connections.ConfigurationMetaNames.DATASOURCE_TYPE: client.connections.get_datasource_type_id_by_name(db_name),
    client.connections.ConfigurationMetaNames.DESCRIPTION: "Connection to external Database",
    client.connections.ConfigurationMetaNames.PROPERTIES: {
        "bucket": bucket_name,
        "api_key": cos_credentials["apikey"],
        "resource_instance_id": cos_credentials["resource_instance_id"],
        "iam_url": "https://iam.cloud.ibm.com/identity/token",
        "url": "https://s3.us.cloud-object-storage.appdomain.cloud"
    }
}

conn_details = client.connections.create(meta_props=conn_meta_props)

```

In [13]:
connection_id = client.connections.get_id(conn_details)

Define connection information to training data.

In [14]:
from ibm_watsonx_ai.helpers import DataConnection, S3Location

credit_risk_conn = DataConnection(
    connection_asset_id=connection_id,
    location=S3Location(bucket=bucket_name, path=filename),
)

training_data_reference = [credit_risk_conn]

Check the connection information. Upload the data and validate.

In [15]:
credit_risk_conn.set_client(client)
credit_risk_conn.write(data=filename, remote_name=filename)
credit_risk_conn.read()

,CheckingStatus,LoanDuration,CreditHistory,LoanPurpose,LoanAmount,ExistingSavings,EmploymentDuration,InstallmentPercent,Sex,OthersOnLoan,...,OwnsProperty,Age,InstallmentPlans,Housing,ExistingCreditsCount,Job,Dependents,Telephone,ForeignWorker,Risk
0,0_to_200,31,credits_paid_to_date,other,1889,100_to_500,less_1,3,female,none,...,savings_insurance,32,none,own,1,skilled,1,none,yes,No Risk
1,less_0,18,credits_paid_to_date,car_new,462,less_100,1_to_4,2,female,none,...,savings_insurance,37,stores,own,2,skilled,1,none,yes,No Risk
2,less_0,15,prior_payments_delayed,furniture,250,less_100,1_to_4,2,male,none,...,real_estate,28,none,own,2,skilled,1,yes,no,No Risk
3,0_to_200,28,credits_paid_to_date,retraining,3693,less_100,greater_7,3,male,none,...,savings_insurance,32,none,own,1,skilled,1,none,yes,No Risk
4,no_checking,28,prior_payments_delayed,education,6235,500_to_1000,greater_7,3,male,none,...,unknown,57,none,own,2,skilled,1,none,yes,Risk
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
244,0_to_200,22,credits_paid_to_date,repairs,812,less_100,4_to_7,3,male,none,...,savings_insurance,19,bank,rent,1,unemployed,1,none,yes,No Risk
245,0_to_200,13,prior_payments_delayed,furniture,2735,500_to_1000,4_to_7,2,female,none,...,savings_insurance,45,stores,rent,2,skilled,1,none,yes,Risk
246,less_0,28,credits_paid_to_date,radio_tv,530,less_100,1_to_4,2,male,none,...,real_estate,32,stores,own,1,skilled,1,none,yes,No Risk
247,less_0,17,credits_paid_to_date,retraining,2119,less_100,1_to_4,3,female,none,...,savings_insurance,23,bank,rent,1,unskilled,1,none,yes,No Risk


### Optimizer configuration

Provide the input information for AutoAI optimizer:
- `name` - experiment name
- `prediction_type` - type of the problem
- `prediction_column` - target column name
- `scoring` - optimization metric

In [16]:
from ibm_watsonx_ai.experiment import AutoAI

experiment = AutoAI(credentials, space_id=space_id)

pipeline_optimizer = experiment.optimizer(
    name="Credit Risk Prediction - AutoAI",
    prediction_type=AutoAI.PredictionType.BINARY,
    prediction_column="Risk",
    include_only_estimators=["RandomForestClassifier"],
)

Configuration parameters can be retrieved via `get_params()`.

In [17]:
pipeline_optimizer.get_params()

{'name': 'Credit Risk Prediction - AutoAI',
 'desc': '',
 'prediction_type': 'binary',
 'prediction_column': 'Risk',
 'prediction_columns': None,
 'timestamp_column_name': None,
 'scoring': None,
 'holdout_size': None,
 'max_num_daub_ensembles': None,
 't_shirt_size': 'l',
 'train_sample_rows_test_size': None,
 'include_only_estimators': [<ClassificationAlgorithms.RF: 'RandomForestClassifier'>],
 'include_batched_ensemble_estimators': None,
 'backtest_num': None,
 'lookback_window': None,
 'forecast_window': None,
 'backtest_gap_length': None,
 'cognito_transform_names': None,
 'csv_separator': ',',
 'excel_sheet': None,
 'encoding': 'utf-8',
 'positive_label': None,
 'drop_duplicates': True,
 'outliers_columns': None,
 'text_processing': None,
 'word2vec_feature_number': None,
 'daub_give_priority_to_runtime': None,
 'text_columns_names': None,
 'sampling_type': None,
 'sample_size_limit': None,
 'sample_rows_limit': None,
 'sample_percentage_limit': None,
 'number_of_batch_rows': Non

<a id="3.-Experiment-run"></a>
## 3. Experiment run

Call the `fit()` method to trigger the AutoAI experiment. You can either use interactive mode (synchronous job) or background mode (asychronous job) by specifying `background_model=True`.

In [18]:
run_details = pipeline_optimizer.fit(
    training_data_reference=training_data_reference, background_mode=False
)

Training job c5bb1f49-573e-49d9-8641-2a35adb5aa35 completed: 100%|████████| [03:03<00:00,  1.84s/it]


You can use the `get_run_status()` method to monitor AutoAI jobs in background mode.

In [19]:
pipeline_optimizer.get_run_status()

'completed'

In [20]:
pipeline_optimizer.summary()

,Enhancements,Estimator,training_roc_auc,holdout_average_precision,holdout_log_loss,training_accuracy,holdout_roc_auc,training_balanced_accuracy,training_f1,holdout_precision,training_average_precision,training_log_loss,holdout_recall,training_precision,holdout_accuracy,holdout_balanced_accuracy,training_recall,holdout_f1
Pipeline Name,,,,,,,,,,,,,,,,,,
Pipeline_1,,RandomForestClassifier,0.816915,0.540392,0.138312,0.754535,0.731618,0.698857,0.826206,0.944444,0.897597,1.043863,1.000000,0.813956,0.96,0.937500,0.839744,0.971429
Pipeline_2,HPO,RandomForestClassifier,0.857733,0.540481,0.526002,0.781261,0.772059,0.760147,0.838123,0.833333,0.933404,0.492098,0.882353,0.863971,0.80,0.753676,0.814103,0.857143
Pipeline_3,"HPO, FE",RandomForestClassifier,0.857885,0.537330,0.518701,0.790150,0.750000,0.773981,0.842970,0.800000,0.927565,0.459822,0.705882,0.876053,0.68,0.665441,0.814103,0.750000
Pipeline_4,"HPO, FE, HPO",RandomForestClassifier,0.863282,0.539113,0.531957,0.785706,0.764706,0.766735,0.840249,0.857143,0.931310,0.458750,0.705882,0.869655,0.72,0.727941,0.814103,0.774194
Pipeline_5,"HPO, FE, HPO, Ensemble",BatchedTreeEnsembleClassifier(RandomForestClas...,0.863282,0.539113,0.531957,0.785706,0.764706,0.766735,0.840249,0.857143,0.931310,0.458750,0.705882,0.869655,0.72,0.727941,0.814103,0.774194


In [21]:
pipeline_name = "Pipeline_1"

In [22]:
pipeline_model = pipeline_optimizer.get_pipeline(
    pipeline_name=pipeline_name, astype=AutoAI.PipelineTypes.ONNX
)

### 3.1. Model evaluation

In [23]:
X_test = credit_risk_conn.read().drop(["Risk"], axis=1)[:3]

In [24]:
X_test_dict = {col: X_test[col].apply(lambda x: [x]).tolist() for col in X_test.columns}
pipeline_model.run([], X_test_dict)

[array(['No Risk', 'No Risk', 'No Risk'], dtype=object),
 [{'No Risk': 0.8999999761581421, 'Risk': 0.10000000149011612},
  {'No Risk': 1.0, 'Risk': 0.0},
  {'No Risk': 0.800000011920929, 'Risk': 0.20000000298023224}]]

<a id="4.-Deploy-and-score"></a>
## 4. Deploy and score

In this section you will learn how to deploy and score pipeline model as webservice using watsonx.ai Runtime instance.

### Online deployment creation

In [25]:
run_id = get_from_json(run_details, ["metadata", "id"])

In [26]:
from ibm_watsonx_ai.deployment import WebService

service = WebService(credentials, source_space_id=space_id)

service.create(
    experiment_run_id=run_id,
    model=pipeline_name,
    deployment_name=f"Credit Risk Deployment AutoAI - ONNX - {pipeline_name}",
)

Preparing an AutoAI Deployment...
Published model uid: 62e3bf7b-37f7-4c38-8e4e-db82ce5a9f79
Deploying model 62e3bf7b-37f7-4c38-8e4e-db82ce5a9f79 using V4 client.


######################################################################################

Synchronous deployment creation for id: '62e3bf7b-37f7-4c38-8e4e-db82ce5a9f79' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
.....
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='368c2d8f-cb12-430f-be1d-0ff493b2d3d5'
-----------------------------------------------------------------------------------------------




Deployment object could be printed to show basic information:

In [27]:
print(service)

To show all available information about the deployment use the `.get_params()` method:

In [28]:
service.get_params()

In [29]:
deployment_id = get_from_json(service.get_params(), ["metadata", "id"])
deployment_id

Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.


'368c2d8f-cb12-430f-be1d-0ff493b2d3d5'

In [30]:
scoring_payload = {"input_data": [{"values": X_test}]}

### Webservice scoring
You can make scoring request by calling `score()` on deployed pipeline.

In [31]:
predictions = client.deployments.score(deployment_id, scoring_payload)

predictions

{'predictions': [{'fields': ['prediction', 'probability'],
   'values': [['No Risk', [0.9, 0.1]],
    ['No Risk', [1.0, 0.0]],
    ['No Risk', [0.8, 0.2]]]}]}

If you want to work with the web service in an external Python application you can retrieve the service object by:
 - Initialize the service by `service = WebService(credentials)`
 - Get deployment_id by `service.list()` method
 - Get webservice object by `service.get('deployment_id')` method

After that you can call `service.score()` method.

### Deleting deployment
You can delete the existing deployment by calling the `service.delete()` command.
To list the existing web services you can use `service.list()`.

<a id="5.-Cleanup"></a>
## 5. Cleanup

If you want to clean up after the notebook execution, i.e. remove any created assets like:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

please follow up this sample [notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="6.-Summary-and-next-steps"></a>
## 6. Summary and next steps

 You successfully completed this notebook! You learned how to use ONNX, scikit-learn machine learning library as well as watsonx.ai Runtime for model creation and deployment. Check out our _[Online Documentation](https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/welcome-main.html?context=wx)_ for more samples, tutorials, documentation, how-tos, and blog posts.

### Authors

**Marta Tomzik**, Software Engineer at watsonx.ai

Copyright © 2025-2026 IBM. This notebook and its source code are released under the terms of the MIT License.